# Practical: CNN vs Fully Connected Neural Network (FCNN)

**Objective:**

Take a $3\times3$ matrix, manually apply CNN operations (Convolution → ReLU → Max Pooling),
then apply a Fully Connected Neural Network on the same matrix, compare the number of
parameters and operations, and explain why CNNs are generally more suitable for image data.

**Note:** All computations are done manually in pure Python — no NumPy, TensorFlow, PyTorch
or any other library is used.

## Step 0: Helper functions

Instead of repeating loops everywhere, the four core operations are written once as small
functions. Each one also counts how many multiplications / additions it performs, so the
cost of the two networks can be compared later.

In [1]:
def show(title, M):
    """Print a matrix row by row."""
    print(title)
    for row in M:
        print("   ", row)
    print()


def convolve(X, K, bias=0, stride=1):
    """Valid 2D convolution, returns (feature_map, multiplications, additions)."""
    n = len(X)
    k = len(K)
    out_size = (n - k) // stride + 1

    feature_map = []
    mults = 0
    adds = 0

    for i in range(out_size):
        row = []
        for j in range(out_size):
            total = 0
            for a in range(k):
                for b in range(k):
                    total += X[i * stride + a][j * stride + b] * K[a][b]
                    mults += 1
            adds += k * k - 1     # adding the k*k products together
            total += bias
            adds += 1             # adding the bias
            row.append(total)
        feature_map.append(row)

    return feature_map, mults, adds


def relu_matrix(M):
    """Apply ReLU element-wise: max(0, value)."""
    return [[max(0, value) for value in row] for row in M]


def max_pool(M, size=2, stride=2):
    """Max pooling, returns (pooled_matrix, comparisons)."""
    out_size = (len(M) - size) // stride + 1
    pooled = []
    comparisons = 0

    for i in range(out_size):
        row = []
        for j in range(out_size):
            best = M[i * stride][j * stride]
            for a in range(size):
                for b in range(size):
                    if (a, b) != (0, 0):
                        comparisons += 1
                        if M[i * stride + a][j * stride + b] > best:
                            best = M[i * stride + a][j * stride + b]
            row.append(best)
        pooled.append(row)

    return pooled, comparisons


def flatten(M):
    """Turn a 2D matrix into a 1D list."""
    return [value for row in M for value in row]


def dense_neuron(x, w, bias):
    """One fully connected neuron: w . x + b. Returns (z, mults, adds)."""
    z = bias
    mults = 0
    adds = 0
    for xi, wi in zip(x, w):
        z += xi * wi
        mults += 1
        adds += 1
    return z, mults, adds

## Step 1: Input matrix

A $3\times3$ matrix is used as a tiny grayscale image. The left column is bright and the
right column is dark, so there is a **vertical edge** in the middle of the image.

In [2]:
X = [
    [3, 0, 1],
    [1, 2, 0],
    [0, 1, 4]
]

show("Input Matrix X (3 x 3):", X)

Input Matrix X (3 x 3):
    [3, 0, 1]
    [1, 2, 0]
    [0, 1, 4]



## Step 2: CNN kernel (filter)

The $2\times2$ kernel below is a **vertical edge detector**. It gives a large positive value
when the left side of a patch is brighter than the right side.

$$K = \begin{bmatrix} 1 & -1 \\ 1 & -1 \end{bmatrix}$$

In [3]:
K = [
    [ 1, -1],
    [ 1, -1]
]

cnn_bias = 0

show("Kernel K (2 x 2) - vertical edge detector:", K)
print("Bias:", cnn_bias)

Kernel K (2 x 2) - vertical edge detector:
    [1, -1]
    [1, -1]

Bias: 0


## Step 3: Convolution (manual)

The kernel slides over the input with stride 1.

Output size $= \dfrac{3 - 2}{1} + 1 = 2$, so the feature map is $2\times2$.

The important point: the **same 4 weights** are reused at every position. This is called
**weight sharing**.

In [4]:
feature_map, conv_mults, conv_adds = convolve(X, K, cnn_bias)

# Show the arithmetic at every position
for i in range(len(feature_map)):
    for j in range(len(feature_map)):
        terms = []
        for a in range(2):
            for b in range(2):
                terms.append(str(X[i + a][j + b]) + "*(" + str(K[a][b]) + ")")
        print("Z[" + str(i) + "][" + str(j) + "] = " + " + ".join(terms)
              + " = " + str(feature_map[i][j]))
print()

show("Feature Map after Convolution:", feature_map)

Z[0][0] = 3*(1) + 0*(-1) + 1*(1) + 2*(-1) = 2
Z[0][1] = 0*(1) + 1*(-1) + 2*(1) + 0*(-1) = 1
Z[1][0] = 1*(1) + 2*(-1) + 0*(1) + 1*(-1) = -2
Z[1][1] = 2*(1) + 0*(-1) + 1*(1) + 4*(-1) = -1

Feature Map after Convolution:
    [2, 1]
    [-2, -1]



## Step 4: ReLU activation

$\text{ReLU}(z) = \max(0, z)$

Negative responses mean "edge in the opposite direction", and ReLU removes them so the
network keeps only the pattern this filter is looking for.

In [5]:
activated = relu_matrix(feature_map)

show("After ReLU:", activated)

After ReLU:
    [2, 1]
    [0, 0]



## Step 5: Max Pooling ($2\times2$, stride 2)

Pooling keeps only the **strongest** response in the window and throws away its exact
position. This is what makes the CNN tolerant to small shifts in the image.

Output size $= \dfrac{2 - 2}{2} + 1 = 1$, so a single value comes out.

In [6]:
pooled, pool_comparisons = max_pool(activated, size=2, stride=2)

show("After Max Pooling:", pooled)

cnn_output = pooled[0][0]
print("Final CNN output (edge strength):", cnn_output)

After Max Pooling:
    [2]

Final CNN output (edge strength): 2


## Step 6: FCNN on the same $3\times3$ matrix

A fully connected network cannot take a 2D matrix, so the **raw image** must be flattened
into a $9\times1$ vector first. Notice the difference:

* CNN flattens the **pooled feature map** (after features are already extracted).
* FCNN flattens the **raw pixels** (before any feature extraction), which destroys the
  spatial structure immediately.

One neuron now needs **one separate weight for every pixel**.

In [7]:
fc_input = flatten(X)
print("Flattened raw input (9 values):", fc_input)
print()

# 9 weights - one for each pixel position
W_fc = [0.5, -1.0, 0.0, 1.0, 0.5, -0.5, 0.0, 1.0, -1.0]
fc_bias = 1.0

z_fc, fc_mults, fc_adds = dense_neuron(fc_input, W_fc, fc_bias)

terms = []
for xi, wi in zip(fc_input, W_fc):
    terms.append(str(xi) + "*(" + str(wi) + ")")
print("z = " + " + ".join(terms) + " + " + str(fc_bias))
print("z =", z_fc)
print()

fcnn_output = max(0, z_fc)
print("FCNN output after ReLU:", fcnn_output)

Flattened raw input (9 values): [3, 0, 1, 1, 2, 0, 0, 1, 4]

z = 3*(0.5) + 0*(-1.0) + 1*(0.0) + 1*(1.0) + 2*(0.5) + 0*(-0.5) + 0*(0.0) + 1*(1.0) + 4*(-1.0) + 1.0
z = 1.5

FCNN output after ReLU: 1.5


## Step 7: Parameter and operation comparison

In [8]:
cnn_params  = (2 * 2) + 1      # 4 kernel weights + 1 bias
fcnn_params = 9 + 1            # 9 pixel weights + 1 bias

print("=" * 52)
print("{:<30}{:>10}{:>10}".format("Metric", "CNN", "FCNN"))
print("=" * 52)
print("{:<30}{:>10}{:>10}".format("Trainable parameters", cnn_params, fcnn_params))
print("{:<30}{:>10}{:>10}".format("Multiplications", conv_mults, fc_mults))
print("{:<30}{:>10}{:>10}".format("Additions (incl. bias)", conv_adds, fc_adds))
print("{:<30}{:>10}{:>10}".format("Comparisons (pooling)", pool_comparisons, 0))
print("{:<30}{:>10}{:>10}".format("Final output", cnn_output, fcnn_output))
print("=" * 52)

Metric                               CNN      FCNN
Trainable parameters                   5        10
Multiplications                       16         9
Additions (incl. bias)                16         9
Comparisons (pooling)                  3         0
Final output                           2       1.5


**Observation:** on such a tiny input the CNN actually performs *more* multiplications
(16 vs 9), because the same kernel is applied at 4 different positions. But it needs only
**half the parameters** (5 vs 10). Parameters are what the network must learn and store, and
Step 9 shows how this gap explodes on real images.

## Step 8: Experiment — shift the pattern

The same bright region is moved one column to the right. A good image model should still
detect it. This is the property called **translation invariance**.

In [9]:
X_shifted = [
    [0, 3, 0],
    [0, 1, 2],
    [0, 0, 1]
]

show("Shifted input:", X_shifted)

# --- CNN on the shifted image ---
fm2, _, _ = convolve(X_shifted, K, cnn_bias)
act2 = relu_matrix(fm2)
pooled2, _ = max_pool(act2, size=2, stride=2)

show("CNN feature map after ReLU:", act2)
print("CNN output   -> original:", cnn_output, "| shifted:", pooled2[0][0])

# --- FCNN on the shifted image ---
z2, _, _ = dense_neuron(flatten(X_shifted), W_fc, fc_bias)
print("FCNN output  -> original:", fcnn_output, "| shifted:", max(0, z2))

Shifted input:
    [0, 3, 0]
    [0, 1, 2]
    [0, 0, 1]

CNN feature map after ReLU:
    [0, 2]
    [0, 0]

CNN output   -> original: 2 | shifted: 2
FCNN output  -> original: 1.5 | shifted: 0


The CNN's kernel finds the edge wherever it appears — the response simply moves to a
different cell of the feature map, and max pooling picks it up anyway.

The FCNN's weights are tied to **fixed pixel positions**, so moving the same pattern gives a
completely different output. It would have to learn the pattern separately for every possible
position in the image.

## Step 9: What happens on a real image?

A $28\times28$ grayscale image (MNIST) and a $224\times224$ colour image are compared below.

In [10]:
H = W = 28
num_filters = 32
k = 3
hidden_neurons = 128

cnn_layer  = num_filters * (k * k * 1 + 1)
fcnn_layer = hidden_neurons * (H * W + 1)

print("28 x 28 grayscale image")
print("  CNN layer  (32 filters of 3x3) :", format(cnn_layer, ","), "parameters")
print("  FCNN layer (128 neurons)       :", format(fcnn_layer, ","), "parameters")
print("  FCNN needs", round(fcnn_layer / cnn_layer), "times more parameters")
print()

print("224 x 224 x 3 colour image")
print("  CNN layer  (32 filters of 3x3) :",
      format(num_filters * (k * k * 3 + 1), ","), "parameters")
print("  FCNN layer (128 neurons)       :",
      format(hidden_neurons * (224 * 224 * 3 + 1), ","), "parameters")
print()
print("The CNN parameter count does not depend on the image size at all,")
print("because the same small kernel is reused everywhere.")

28 x 28 grayscale image
  CNN layer  (32 filters of 3x3) : 320 parameters
  FCNN layer (128 neurons)       : 100,480 parameters
  FCNN needs 314 times more parameters

224 x 224 x 3 colour image
  CNN layer  (32 filters of 3x3) : 896 parameters
  FCNN layer (128 neurons)       : 19,267,712 parameters

The CNN parameter count does not depend on the image size at all,
because the same small kernel is reused everywhere.


## Step 10: Comparison summary

| Aspect | CNN | FCNN |
|---|---|---|
| Input handling | Keeps the $3\times3$ grid structure | Raw image flattened to $9\times1$ |
| Connectivity | Local — each output sees one $2\times2$ patch | Global — each neuron sees all 9 pixels |
| Weight sharing | Yes — same 4 weights reused everywhere | No — one weight per pixel position |
| Parameters (this example) | 5 | 10 |
| Parameters grow with image size? | No | Yes, grows with $H \times W$ |
| Spatial information | Preserved until pooling | Lost at the input |
| Shifted pattern | Still detected | Output changes completely |
| Operations used | Convolution → ReLU → Max Pooling | Dot product → ReLU |

## Conclusion

CNNs are more suitable for image data because:

1. **Spatial structure is preserved** — convolution works directly on the 2D grid, so
   neighbouring pixels stay neighbours.
2. **Local receptive fields** — each kernel looks at a small patch and learns local
   features such as edges, corners and textures.
3. **Weight sharing** — the same kernel is reused at every position, so the parameter
   count stays small and is **independent of the image size**.
4. **Translation invariance** — with pooling, a feature is detected wherever it appears
   in the image (shown in Step 8).
5. **Fewer parameters → less overfitting** and much lower memory cost on real images
   (shown in Step 9).

An FCNN flattens the raw image before learning anything, so it throws away spatial
information, needs a separate weight for every pixel, and its parameter count grows with
the image size — which makes it expensive and prone to overfitting on image data.

*Md. Abu Sayem*

*Roll: 25/SET/MTCE/014*